In [1]:
import pandas as pd

In [2]:
climate_clean = pd.read_csv(r'Interim\climate_clean.csv')
fuelprice_clean = pd.read_csv(r'Interim\fuelprice_clean.csv')
ttc_ridership = pd.read_csv(r'Interim\ttc_ridership.csv')
unemployment_clean = pd.read_csv(r'Interim\unemployment_clean.csv')

In [3]:
print("ttc duplicate dates:", ttc_ridership["date"].duplicated().sum())
print("climate duplicate dates:", climate_clean["date"].duplicated().sum())
print("fuel duplicate dates:", fuelprice_clean["date"].duplicated().sum())
print("unemp duplicate dates:", unemployment_clean["date"].duplicated().sum())

ttc duplicate dates: 0
climate duplicate dates: 0
fuel duplicate dates: 0
unemp duplicate dates: 0


In [4]:
main_df = (
    ttc_ridership
    .merge(climate_clean, on= 'date' , how= 'left')
    .merge(fuelprice_clean, on='date' , how='left')
    .merge(unemployment_clean, on='date', how='left')
)

main_df.head()

,date,year_x,month_x,ridership,avg_temp,avg_max_temp,avg_min_temp,total_rain_mm,total_snow_cm,total_precip_mm,...,missing_precip_days,missing_snow_days,temp_completeness,precip_completeness,snow_completeness,climate_data_flag,Unnamed: 0,gas_price_regular,GEO,Unemployment Rate
0,2007-01-01,2007,1,1400000,-3.100000,0.387097,-6.564516,24.0,16.4,45.7,...,0,0,1.0,1.0,1.0,False,0,80.1,NaN,NaN
1,2007-02-01,2007,2,1486000,-7.675000,-3.875000,-11.446429,0.0,34.4,25.0,...,0,0,1.0,1.0,1.0,False,1,89.3,NaN,NaN
2,2007-03-01,2007,3,1485000,0.354839,5.274194,-4.548387,24.2,24.8,48.9,...,0,0,1.0,1.0,1.0,False,2,101.1,NaN,NaN
3,2007-04-01,2007,4,1491000,6.236667,10.933333,1.500000,70.3,2.6,73.0,...,0,0,1.0,1.0,1.0,False,3,101.1,NaN,NaN
4,2007-05-01,2007,5,1474000,13.964516,21.064516,6.822581,63.6,0.0,63.6,...,0,0,1.0,1.0,1.0,False,4,105.5,NaN,NaN


In [5]:
main_df.dtypes

date                    object
year_x                   int64
month_x                  int64
ridership                int64
avg_temp               float64
avg_max_temp           float64
avg_min_temp           float64
total_rain_mm          float64
total_snow_cm          float64
total_precip_mm        float64
year_y                   int64
month_y                  int64
days_in_month            int64
missing_temp_days        int64
missing_precip_days      int64
missing_snow_days        int64
temp_completeness      float64
precip_completeness    float64
snow_completeness      float64
climate_data_flag         bool
Unnamed: 0               int64
gas_price_regular      float64
GEO                     object
Unemployment Rate      float64
dtype: object

In [6]:
main_df = main_df.drop(columns=['year_x','month_x', 'Unnamed: 0'])


In [7]:
main_df = main_df.rename(columns={'year_y':'year' , 'month_y':'month'})

In [8]:
main_df.dtypes

date                    object
ridership                int64
avg_temp               float64
avg_max_temp           float64
avg_min_temp           float64
total_rain_mm          float64
total_snow_cm          float64
total_precip_mm        float64
year                     int64
month                    int64
days_in_month            int64
missing_temp_days        int64
missing_precip_days      int64
missing_snow_days        int64
temp_completeness      float64
precip_completeness    float64
snow_completeness      float64
climate_data_flag         bool
gas_price_regular      float64
GEO                     object
Unemployment Rate      float64
dtype: object

In [9]:
main_df['date'] = pd.to_datetime(main_df['date'] , format='%Y-%m-%d')

In [10]:
# date flags
main_df['quarter'] = main_df['date'].dt.quarter

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"

main_df["season"] = main_df["month"].apply(get_season)

In [11]:
def get_covid_period(date):
    if date < pd.Timestamp("2020-03-01"):
        return "Pre-COVID"
    elif date <= pd.Timestamp("2022-12-01"):
        return "COVID/Recovery"
    else:
        return "Post-COVID"

main_df["covid_period"] = main_df["date"].apply(get_covid_period)

In [12]:
main_df.head()

,date,ridership,avg_temp,avg_max_temp,avg_min_temp,total_rain_mm,total_snow_cm,total_precip_mm,year,month,...,temp_completeness,precip_completeness,snow_completeness,climate_data_flag,gas_price_regular,GEO,Unemployment Rate,quarter,season,covid_period
0,2007-01-01,1400000,-3.100000,0.387097,-6.564516,24.0,16.4,45.7,2007,1,...,1.0,1.0,1.0,False,80.1,NaN,NaN,1,Winter,Pre-COVID
1,2007-02-01,1486000,-7.675000,-3.875000,-11.446429,0.0,34.4,25.0,2007,2,...,1.0,1.0,1.0,False,89.3,NaN,NaN,1,Winter,Pre-COVID
2,2007-03-01,1485000,0.354839,5.274194,-4.548387,24.2,24.8,48.9,2007,3,...,1.0,1.0,1.0,False,101.1,NaN,NaN,1,Spring,Pre-COVID
3,2007-04-01,1491000,6.236667,10.933333,1.500000,70.3,2.6,73.0,2007,4,...,1.0,1.0,1.0,False,101.1,NaN,NaN,2,Spring,Pre-COVID
4,2007-05-01,1474000,13.964516,21.064516,6.822581,63.6,0.0,63.6,2007,5,...,1.0,1.0,1.0,False,105.5,NaN,NaN,2,Spring,Pre-COVID


In [13]:
print("Duplicate dates in main_df:", main_df["date"].duplicated().sum())

print("\nMissing values:")
print(main_df.isna().sum())

print("\nDate range:")
print(main_df["date"].min(), "to", main_df["date"].max())

Duplicate dates in main_df: 0

Missing values:
date                    0
ridership               0
avg_temp                0
avg_max_temp            0
avg_min_temp            0
total_rain_mm           0
total_snow_cm           0
total_precip_mm         0
year                    0
month                   0
days_in_month           0
missing_temp_days       0
missing_precip_days     0
missing_snow_days       0
temp_completeness       0
precip_completeness     0
snow_completeness       0
climate_data_flag       0
gas_price_regular       0
GEO                    48
Unemployment Rate      48
quarter                 0
season                  0
covid_period            0
dtype: int64

Date range:
2007-01-01 00:00:00 to 2025-12-01 00:00:00


In [14]:
main_df_end = main_df[
    [
        "date",
        "year",
        "month",
        "quarter",
        "season",
        "covid_period",
        "ridership",
        "gas_price_regular",
        "Unemployment Rate",
        "avg_temp",
        "total_precip_mm",
        "total_snow_cm",
        "avg_max_temp",
        "avg_min_temp",
        "climate_data_flag",
        "days_in_month",
        "missing_temp_days",
        "missing_precip_days",
        "missing_snow_days",
        "temp_completeness",
        "precip_completeness",
        "snow_completeness"
    ]
]

In [15]:
main_df_end = main_df_end.rename(columns={'Unemployment Rate': 'unemployment_rate'})

In [16]:
main_df_end.dtypes

date                   datetime64[ns]
year                            int64
month                           int64
quarter                         int32
season                         object
covid_period                   object
ridership                       int64
gas_price_regular             float64
unemployment_rate             float64
avg_temp                      float64
total_precip_mm               float64
total_snow_cm                 float64
avg_max_temp                  float64
avg_min_temp                  float64
climate_data_flag                bool
days_in_month                   int64
missing_temp_days               int64
missing_precip_days             int64
missing_snow_days               int64
temp_completeness             float64
precip_completeness           float64
snow_completeness             float64
dtype: object

In [20]:
main_df_end.to_csv(r'3_Processed_Data\main_dataset_2.csv')

In [21]:
main_df_end.to_excel(r'3_Processed_Data\main_dataset.xlsx')